In [ ]:


from google.colab import drive
drive.mount('/content/drive')

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
import pandas as pd

# ── paths
base_path   = '/content/drive/MyDrive/Project Agri/Dataset'
field1_path = os.path.join(base_path, 'Field Images/FieldImages-1')
field2_path = os.path.join(base_path, 'Field Images/FieldImages-2')
lab_path    = os.path.join(base_path, 'Lab Images')

Mounted at /content/drive


In [ ]:

disease_cols = ['GLS','NCLB','PLS','CR','SR','NoFoliarSymptoms','Other','UnidentifiedDisease']
df1 = pd.read_csv(os.path.join(field1_path, 'Corn-Annotations.csv'))
df1['label_count'] = df1[disease_cols].sum(axis=1)
df1 = df1[df1['label_count'] == 1]
def get_label(row):
    for col in disease_cols:
        if row[col] == 1: return col
df1['label'] = df1.apply(get_label, axis=1)
df1['image_path'] = df1['filePath'].apply(lambda x: os.path.join(field1_path, 'leaf_images', x))
df1['source'] = 'field'
df1 = df1[['image_path','label','source']]

data_field2 = []
for label in os.listdir(field2_path):
    cp = os.path.join(field2_path, label)
    if os.path.isdir(cp):
        for img in os.listdir(cp):
            data_field2.append([os.path.join(cp, img), label.upper(), 'field'])
df2 = pd.DataFrame(data_field2, columns=['image_path','label','source'])

data_lab = []
for label in os.listdir(lab_path):
    cp = os.path.join(lab_path, label)
    if os.path.isdir(cp):
        for img in os.listdir(cp)[:200]:
            data_lab.append([os.path.join(cp, img), label, 'lab'])
df3 = pd.DataFrame(data_lab, columns=['image_path','label','source'])

label_map = {
    'GLS':'Gray_Leaf_Spot','Corn_Cercospora_leaf_spot_Gray_leaf_spot':'Gray_Leaf_Spot',
    'NCLB':'Northern_Leaf_Blight','NLB':'Northern_Leaf_Blight','NLS':'Northern_Leaf_Blight',
    'Corn_Northern_Leaf_Blight':'Northern_Leaf_Blight',
    'CR':'Common_Rust','Corn_Common_rust':'Common_Rust',
    'NoFoliarSymptoms':'Healthy','Corn_healthy':'Healthy',
}
def standardise(x):
    x = str(x).strip().replace('___','_').replace(' ','_')
    return label_map.get(x, x)
for d in [df1, df2, df3]:
    d['label'] = d['label'].apply(standardise)

df_all = pd.concat([df1, df2, df3], ignore_index=True)
TARGET_CLASSES = ['Gray_Leaf_Spot','Northern_Leaf_Blight','Common_Rust','Healthy']
df_all = df_all[df_all['label'].isin(TARGET_CLASSES)].reset_index(drop=True)
print('Dataset ready:', df_all['label'].value_counts().to_dict())

Dataset ready: {'Northern_Leaf_Blight': 1388, 'Gray_Leaf_Spot': 1353, 'Healthy': 432, 'Common_Rust': 307}


In [ ]:

def gaussian_blur(img, intensity):
    return img.filter(ImageFilter.GaussianBlur(radius=intensity))

def brightness_shift(img, intensity):
    return ImageEnhance.Brightness(img).enhance(1 + intensity)

def occlusion(img, intensity):
    img = img.copy()
    w, h = img.size
    draw = ImageDraw.Draw(img)
    fraction = min(intensity / 10, 0.9)
    area = int(w * h * fraction)
    covered = 0
    rng = random.Random(42)
    while covered < area:
        rw = rng.randint(20, int(w*0.2)); rh = rng.randint(20, int(h*0.2))
        x = rng.randint(0, w-rw);        y  = rng.randint(0, h-rh)
        draw.rectangle([x, y, x+rw, y+rh], fill=(0,0,0))
        covered += rw * rh
    return img

def rotation(img, intensity):
    return img.rotate(intensity)

NOISE_FUNCS = {
    'Gaussian Blur':    (gaussian_blur,    [1, 3, 5, 9, 15]),
    'Brightness Shift': (brightness_shift, [-0.9, -0.6, 0.6, 0.9, 2.0]),
    'Occlusion':        (occlusion,        [1, 3, 5, 8, 10]),
    'Rotation':         (rotation,         [15, 30, 45, 90, 100]),
}

In [ ]:


DISPLAY_CLASS = 'Northern_Leaf_Blight'   # ← change to any of the 4 classes

sample_path = df_all[df_all['label'] == DISPLAY_CLASS].sample(1, random_state=7).iloc[0]['image_path']
sample_img  = Image.open(sample_path).convert('RGB').resize((224, 224))

noise_names  = list(NOISE_FUNCS.keys())
n_noise      = len(noise_names)
n_intensities= 5

fig, axes = plt.subplots(
    n_noise, n_intensities + 1,           # +1 for the original
    figsize=(3 * (n_intensities + 1), 3 * n_noise)
)
fig.suptitle(f'Noise Visualisation — {DISPLAY_CLASS.replace("_"," ")}',
             fontsize=16, fontweight='bold', y=1.01)

for row_idx, noise_name in enumerate(noise_names):
    fn, intensities = NOISE_FUNCS[noise_name]

    # Original image in first column
    axes[row_idx, 0].imshow(sample_img)
    axes[row_idx, 0].set_title('Original', fontsize=9)
    axes[row_idx, 0].axis('off')
    axes[row_idx, 0].set_ylabel(noise_name, fontsize=11, fontweight='bold', rotation=90, labelpad=10)

    for col_idx, intensity in enumerate(intensities):
        noisy = fn(sample_img.copy(), intensity)
        axes[row_idx, col_idx + 1].imshow(noisy)
        axes[row_idx, col_idx + 1].set_title(f'I = {intensity}', fontsize=9)
        axes[row_idx, col_idx + 1].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Project Agri/noise_visualisation_single_class.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved!')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── Plot 2: All 4 disease classes — one noise type at all intensities ──────────
# Run this cell 4 times changing DISPLAY_NOISE to see each noise type

DISPLAY_NOISE = 'Brightness Shift'   # ← change to: 'Brightness Shift', 'Occlusion', 'Rotation' , 'Gaussian Blur'

fn, intensities = NOISE_FUNCS[DISPLAY_NOISE]

# Pick one sample per class
sample_images = {}
for cls in TARGET_CLASSES:
    path = df_all[df_all['label'] == cls].sample(1, random_state=7).iloc[0]['image_path']
    sample_images[cls] = Image.open(path).convert('RGB').resize((224, 224))

n_classes    = len(TARGET_CLASSES)
n_cols       = len(intensities) + 1   # +1 for original

fig, axes = plt.subplots(n_classes, n_cols, figsize=(3 * n_cols, 3 * n_classes))
fig.suptitle(f'Noise: {DISPLAY_NOISE} — All Disease Classes',
             fontsize=16, fontweight='bold', y=1.01)

for row_idx, cls in enumerate(TARGET_CLASSES):
    img = sample_images[cls]

    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title('Original', fontsize=9)
    axes[row_idx, 0].set_ylabel(cls.replace('_',' '), fontsize=10,
                                 fontweight='bold', rotation=90, labelpad=10)
    axes[row_idx, 0].axis('off')

    for col_idx, intensity in enumerate(intensities):
        noisy = fn(img.copy(), intensity)
        axes[row_idx, col_idx + 1].imshow(noisy)
        axes[row_idx, col_idx + 1].set_title(f'I = {intensity}', fontsize=9)
        axes[row_idx, col_idx + 1].axis('off')

plt.tight_layout()
save_name = DISPLAY_NOISE.replace(' ','_').lower()
plt.savefig(f'/content/drive/MyDrive/Project Agri/noise_visualisation_{save_name}.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved!')

Output hidden; open in https://colab.research.google.com to view.